
# 3I/ATLAS Research Notebook
This notebook pulls **JPL Horizons ephemerides** and **MAST** search results for **3I/ATLAS (C/2025 N1 (ATLAS))**, builds a tidy dataset catalog (CSV/SQLite), and creates quick‑look plots for **trajectory** and **brightness timeline**.


In [5]:

# Parameters
OBJECT_NAMES = ["C/2025 N1"]  # Correct JPL Horizons identifier (without (ATLAS))

# Rolling 7-day window (relative dates)
today = datetime.now()
DATE_END = today.strftime("%Y-%m-%d")
DATE_START = (today - timedelta(days=7)).strftime("%Y-%m-%d")

STEP = "1d"  # ephemerides cadence

# Output paths
OUT_DIR = "outputs_3I_ATLAS"
os.makedirs(OUT_DIR, exist_ok=True)

CSV_EPHEMERIDES = os.path.join(OUT_DIR, "ephemerides.csv")
CSV_MAST        = os.path.join(OUT_DIR, "mast_catalog.csv")
CSV_DATASETS    = os.path.join(OUT_DIR, "dataset_catalog.csv")
SQLITE_DB       = os.path.join(OUT_DIR, "catalog.sqlite")
FIG_TRAJECTORY  = os.path.join(OUT_DIR, "trajectory_ra_dec.png")
FIG_BRIGHTNESS  = os.path.join(OUT_DIR, "brightness_timeline.png")


In [1]:

# Imports (with optional astroquery)
import os
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

# Try astroquery for Horizons + MAST
HAVE_ASTROQUERY = True
try:
    from astroquery.jplhorizons import Horizons
    from astroquery.mast import Observations
except Exception as e:
    HAVE_ASTROQUERY = False
    print("astroquery not available; will use offline fallbacks.")


## Pull JPL Horizons Ephemerides

In [6]:

def fetch_ephemerides(object_names, date_start, date_end, step):
    if not HAVE_ASTROQUERY:
        raise RuntimeError("astroquery is not available. Cannot fetch ephemerides.")
    
    errors = []
    for name in object_names:
        try:
            obj = Horizons(id=name, location='500@10',  # heliocentric frame proxy: use Sun barycenter? Using Earth L1 (500@10) is fine for apparent RA/Dec
                           epochs={'start': date_start, 'stop': date_end, 'step': step})
            eph = obj.ephemerides()
            if len(eph) > 0:
                df = eph.to_pandas()
                df['object_name'] = name
                print(f"Successfully retrieved ephemerides for {name}")
                return df
        except Exception as e:
            error_msg = f"Failed to query {name}: {e}"
            print(error_msg)
            errors.append(error_msg)
            continue
    
    # If we get here, all queries failed
    raise RuntimeError(f"Could not retrieve ephemerides for any object. Errors:\n" + "\n".join(errors))

ephem_df = fetch_ephemerides(OBJECT_NAMES, DATE_START, DATE_END, STEP)
ephem_df.to_csv(CSV_EPHEMERIDES, index=False)
ephem_df.head()


Successfully retrieved ephemerides for C/2025 N1


,targetname,datetime_str,datetime_jd,M1,solar_presence,k1,interfering_body,RA,DEC,RA_app,...,r_rate_3sigma,SBand_3sigma,XBand_3sigma,DoppDelay_3sigma,true_anom,hour_angle,alpha_true,PABLon,PABLat,object_name
0,ATLAS (C/2025 N1),2025-Nov-02 00:00,2460981.5,12.3,,4.5,,188.92806,-0.01791,188.09054,...,0.026423,371.51,1350.23,0.354392,5.8573,<NA>,0.0130,188.2154,3.5232,C/2025 N1
1,ATLAS (C/2025 N1),2025-Nov-03 00:00,2460982.5,12.3,,4.5,,187.37789,0.53998,186.44975,...,0.027223,378.17,1374.44,0.364035,7.5056,<NA>,0.0130,186.5678,3.4244,C/2025 N1
2,ATLAS (C/2025 N1),2025-Nov-04 00:00,2460983.5,12.3,,4.5,,185.83747,1.09387,184.81899,...,0.028009,384.35,1396.92,0.374030,9.1432,<NA>,0.0129,184.9312,3.3234,C/2025 N1
3,ATLAS (C/2025 N1),2025-Nov-05 00:00,2460984.5,12.3,,4.5,,184.30855,1.64263,183.20041,...,0.028781,390.06,1417.67,0.384363,10.7680,<NA>,0.0129,183.3078,3.2205,C/2025 N1
4,ATLAS (C/2025 N1),2025-Nov-06 00:00,2460985.5,12.3,,4.5,,182.79280,2.18522,181.59606,...,0.029537,395.30,1436.70,0.395021,12.3779,<NA>,0.0128,181.6996,3.1160,C/2025 N1


## Query MAST for HST/JWST Observations

In [7]:

def query_mast(object_names, date_start, date_end, max_records=100, timeout=60):
    if not HAVE_ASTROQUERY:
        raise RuntimeError("astroquery is not available. Cannot query MAST.")
    
    # Set timeout for astroquery
    from astroquery.mast import Conf
    Conf.timeout = timeout
    
    # Try by target name; also restrict to HST/JWST missions
    all_obs = []
    errors = []
    
    for i, name in enumerate(object_names):
        print(f"Querying MAST for object {i+1}/{len(object_names)}: {name}")
        print(f"  Date range: {date_start} to {date_end}")
        print(f"  Timeout: {timeout}s, Max records: {max_records}")
        
        try:
            # Add timeout and page size limits
            obs = Observations.query_criteria(
                target_name=name,
                obs_collection=["HST", "JWST"],
                dataproduct_type=["image","spectrum"],
                t_min=[date_start, None],
                t_max=[None, date_end],
                pagesize=max_records  # Limit results per query
            )
            
            print(f"  -> Found {len(obs)} observations for {name}")
            
            if len(obs) > 0:
                all_obs.append(obs.to_pandas())
        except Exception as e:
            error_msg = f"MAST query failed for {name}: {e}"
            print(error_msg)
            errors.append(error_msg)
            continue

    if all_obs:
        df = pd.concat(all_obs, ignore_index=True)
        print(f"\nTotal observations across all objects: {len(df)}")
        return df
    else:
        # If we get here, no observations were found
        raise RuntimeError(f"No MAST observations found for any object. Errors:\n" + "\n".join(errors) if errors else "No observations matched the query criteria.")

mast_df = query_mast(OBJECT_NAMES, DATE_START, DATE_END, max_records=50, timeout=120)
mast_df.to_csv(CSV_MAST, index=False)
mast_df.head()


Querying MAST for object 1/1: C/2025 N1
  Date range: 2025-11-02 to 2025-11-09
  Timeout: 120s, Max records: 50
MAST query failed for C/2025 N1: Thread was being aborted.
MAST query failed for C/2025 N1: Thread was being aborted.


RuntimeError: No MAST observations found for any object. Errors:
MAST query failed for C/2025 N1: Thread was being aborted.

## Build Tidy Dataset Catalog (CSV + SQLite)

In [ ]:

# Normalize/prepare
eph_min = ephem_df[['datetime_str', 'RA', 'DEC', 'r', 'delta', 'V', 'object_name']].copy()
eph_min.rename(columns={'datetime_str':'datetime_utc'}, inplace=True)
eph_min['source'] = 'JPL Horizons'

mast_min = mast_df[['obsid','obs_collection','instrument_name','target_name','t_observe_start','t_observe_end','filters','proposal_id','dataURL']].copy()
mast_min.rename(columns={
    't_observe_start':'datetime_obs_start_utc',
    't_observe_end':'datetime_obs_end_utc',
    'obs_collection':'mission'
}, inplace=True)
mast_min['source'] = 'MAST'

# Persist as separate CSVs (already saved ephemerides & mast above)
# Build a combined catalog index for convenience
dataset_catalog = {
    'ephemerides_csv': CSV_EPHEMERIDES,
    'mast_csv': CSV_MAST
}
catalog_df = pd.DataFrame([dataset_catalog])
catalog_df.to_csv(CSV_DATASETS, index=False)

# Save to SQLite
conn = sqlite3.connect(SQLITE_DB)
eph_min.to_sql("ephemerides", conn, if_exists="replace", index=False)
mast_min.to_sql("mast_observations", conn, if_exists="replace", index=False)
conn.commit()
conn.close()

print("Saved:")
print(" -", CSV_EPHEMERIDES)
print(" -", CSV_MAST)
print(" -", CSV_DATASETS)
print(" -", SQLITE_DB)


Saved:
 - outputs_3I_ATLAS\ephemerides.csv
 - outputs_3I_ATLAS\mast_catalog.csv
 - outputs_3I_ATLAS\dataset_catalog.csv
 - outputs_3I_ATLAS\catalog.sqlite


## Quick‑Look Plots

In [8]:

# Ensure datetime conversion
eph_plot = ephem_df.copy()
if 'datetime_str' in eph_plot.columns:
    eph_plot['dt'] = pd.to_datetime(eph_plot['datetime_str'])
elif 'datetime_utc' in eph_plot.columns:
    eph_plot['dt'] = pd.to_datetime(eph_plot['datetime_utc'])
else:
    # Horizons pandas may include 'datetime_str' already; default fallback
    eph_plot['dt'] = pd.to_datetime(eph_plot.iloc[:,0], errors='coerce')

# 1) Trajectory: RA vs Dec
plt.figure()
plt.plot(eph_plot['RA'], eph_plot['DEC'], marker='o')
plt.xlabel("Right Ascension (deg)")
plt.ylabel("Declination (deg)")
plt.title("3I/ATLAS Trajectory (Sky Plane)")
plt.gca().invert_xaxis()  # RA increases to the left convention
plt.grid(True)
plt.tight_layout()
plt.savefig(FIG_TRAJECTORY, dpi=150)
plt.close()

# 2) Brightness timeline: V magnitude vs time
plt.figure()
plt.plot(eph_plot['dt'], eph_plot['V'], marker='o')
plt.gca().invert_yaxis()  # lower magnitudes = brighter
plt.xlabel("Date (UTC)")
plt.ylabel("V Magnitude")
plt.title("3I/ATLAS Brightness Timeline (V)")
plt.grid(True)
plt.tight_layout()
plt.savefig(FIG_BRIGHTNESS, dpi=150)
plt.close()

FIG_TRAJECTORY, FIG_BRIGHTNESS


('outputs_3I_ATLAS\\trajectory_ra_dec.png',
 'outputs_3I_ATLAS\\brightness_timeline.png')

## Notes
- When you have internet access, re‑run all cells to pull live data.
- To refine MAST queries, adjust `Observations.query_criteria` with specific missions/instruments or RA/Dec cones.

---

# Post-publication update (2025–2026)

The discovery-era cells above were authored when 3I/ATLAS was a faint sky position with a single MPC orbit. Between July 2025 and May 2026, roughly **100 peer-reviewed and preprint papers** have characterised this object across optical, near-infrared, mid-infrared, sub-millimetre, polarimetric, and radio bands. The picture has converged, and it is more interesting than the popular-press framing.

This section adds two contributions on top of the literature:

- **Part B — cross-interstellar comparison.** We assemble the three known interstellar objects (1I/ʻOumuamua, 2I/Borisov, 3I/ATLAS) into a single normalised table with full provenance — every numeric cell carries the citation that supplied it. We then place the D/H ratios on a single chart against Solar-System cometary families and protosolar / interstellar-medium reference points. The chart does not appear in any single paper; only the synthesis does.
- **Part C — sourced audit of popular-press claims.** A widely-circulated article presented five claims. We grade each against the primary literature with arXiv IDs and DOIs. Three are confirmed, one is wrong, and one is unsupported and contradicted by three independent technosignature null results.

The intent is not summary — it is **synthesis with provenance**. Every number is traceable to a paper; every editorial verdict is sourced.

> **Leadership Insights — Credibility is what you decline to publish.**
>
> The temptation when you run a research site is to capture attention — to repeat the loudest version of every story. The discipline is the opposite: read the article, *pause*, search the primary literature, and keep the items the literature actually supports. Then publish what is true and refuse the rest, plainly, with citations. A reader can tell the difference between a site that prints everything and a site that prints what it has checked — and the second one is the one they come back to.


## Part B — Cross-interstellar comparison

We build a **long-format** DataFrame with one row per `(object, metric)` pair, carrying the citation that supplied the number. Long format is deliberate: it makes provenance per-cell rather than per-column, which is the honest representation for a curated literature snapshot.

The comparison is then pivoted to wide format for human reading and exported to `outputs_3I_ATLAS/interstellar_comparison.csv`.

For 1I/ʻOumuamua, many cometary metrics are *physically absent* (no detected coma, no measurable gas-phase volatiles) rather than missing data — these are marked `n/a (no coma)` so the table tells the truth about why a cell is empty.


In [ ]:
# Curated literature snapshot — cross-interstellar comparison
# Format: (object, metric, value, unit, source_label, source_arxiv_or_doi)
# Every numeric cell is sourced. Where a metric is not physically applicable
# (e.g. cometary gas in 1I/Oumuamua, which had no detected coma), the value
# is the literal string "n/a (no coma)" so the table cannot be mistaken for
# missing data.

import os
import pandas as pd

OUT_DIR = "outputs_3I_ATLAS"
os.makedirs(OUT_DIR, exist_ok=True)

COMPARISON_ROWS = [
    # ---- 1I/'Oumuamua ----
    ("1I/'Oumuamua", "discovery_year",          2017,     "yr",    "Meech et al. 2017",                "10.1038/nature25020"),
    ("1I/'Oumuamua", "v_inf_km_s",              26.33,    "km/s",  "Meech et al. 2017",                "10.1038/nature25020"),
    ("1I/'Oumuamua", "perihelion_au",           0.255,    "AU",    "Meech et al. 2017",                "10.1038/nature25020"),
    ("1I/'Oumuamua", "eccentricity",            1.20,     "—",     "Meech et al. 2017",                "10.1038/nature25020"),
    ("1I/'Oumuamua", "coma_detected",           "no",     "—",     "Meech et al. 2017",                "10.1038/nature25020"),
    ("1I/'Oumuamua", "nongrav_accel_detected",  "yes",    "—",     "Micheli et al. 2018 Nature",       "10.1038/s41586-018-0254-4"),
    ("1I/'Oumuamua", "water_d_h",               "n/a (no coma)",  "—",   "—",                          ""),
    ("1I/'Oumuamua", "methane_d_h",             "n/a (no coma)",  "—",   "—",                          ""),
    ("1I/'Oumuamua", "co2_h2o_ratio",           "n/a (no coma)",  "—",   "—",                          ""),
    ("1I/'Oumuamua", "c12_c13_in_cn",           "n/a (no coma)",  "—",   "—",                          ""),
    ("1I/'Oumuamua", "nucleus_size_m",          "100–1000 (elongated)", "m", "Meech et al. 2017",       "10.1038/nature25020"),

    # ---- 2I/Borisov ----
    ("2I/Borisov",   "discovery_year",          2019,     "yr",    "Guzik et al. 2020 Nature Astron.", "10.1038/s41550-019-0931-8"),
    ("2I/Borisov",   "v_inf_km_s",              32.2,     "km/s",  "Guzik et al. 2020 Nature Astron.", "10.1038/s41550-019-0931-8"),
    ("2I/Borisov",   "perihelion_au",           2.006,    "AU",    "JPL Horizons (orbital solution)",  ""),
    ("2I/Borisov",   "eccentricity",            3.36,     "—",     "JPL Horizons (orbital solution)",  ""),
    ("2I/Borisov",   "coma_detected",           "yes",    "—",     "Bodewits et al. 2020 Nature Astron.", "10.1038/s41550-020-1095-2"),
    ("2I/Borisov",   "nongrav_accel_detected",  "yes",    "—",     "de la Fuente Marcos+ 2020",       ""),
    ("2I/Borisov",   "water_d_h",               "see notes",  "—", "Aravind et al. (limit only)",      ""),
    ("2I/Borisov",   "methane_d_h",             "not measured",  "—", "—",                             ""),
    ("2I/Borisov",   "co_h2o_ratio",            1.7,      "—",     "Bodewits et al. 2020 Nature Astron.", "10.1038/s41550-020-1095-2"),
    ("2I/Borisov",   "co2_h2o_ratio",           "see notes",  "—", "—",                                ""),
    ("2I/Borisov",   "c12_c13_in_cn",           "≈100 (solar-like)",  "—", "Manzini et al. 2020",      ""),
    ("2I/Borisov",   "nucleus_size_m",          "200–500",  "m",   "Jewitt et al. 2020",               ""),

    # ---- 3I/ATLAS ----
    ("3I/ATLAS",     "discovery_year",          2025,     "yr",    "ATLAS survey / MPC",               ""),
    ("3I/ATLAS",     "v_inf_km_s",              58.0,     "km/s",  "Ahuja & Ganesh 2025 ApJL 995 L13", "arXiv:2511.16247"),
    ("3I/ATLAS",     "perihelion_au",           1.357,    "AU",    "JPL Horizons (orbital solution)",  ""),
    ("3I/ATLAS",     "perihelion_date",         "2025-10-29", "UT","JPL Horizons / Biver et al. 2026 A&A 708 L16", "arXiv:2603.23240"),
    ("3I/ATLAS",     "eccentricity",            "see notes (>1)", "—", "JPL Horizons (orbital solution)", ""),
    ("3I/ATLAS",     "coma_detected",           "yes",    "—",     "Roth et al. 2026 JWST",            "arXiv:2603.20460"),
    ("3I/ATLAS",     "nongrav_accel_detected",  "yes",    "—",     "Spada, Królikowska, Dones 2026",   "arXiv:2603.00782"),
    ("3I/ATLAS",     "water_d_h",               "> 6.6e-3",  "—",  "Salazar Manzano et al. 2026 Nat. Astron. (accepted)", "arXiv:2603.07026"),
    ("3I/ATLAS",     "methane_d_h",             0.0331,   "—",     "Roth, Cordiner et al. 2026 JWST",  "arXiv:2603.20445"),
    ("3I/ATLAS",     "co2_h2o_ratio",           "elevated (CO2-dominated)", "—", "Roth et al. 2026 JWST NIRSpec",  "arXiv:2603.20460"),
    ("3I/ATLAS",     "c12_c13_in_cn",           147,      "—",     "Opitom et al. 2026 (submitted Nature)", "arXiv:2603.07187"),
    ("3I/ATLAS",     "n14_n15_in_cn",           "measured", "—",   "Opitom et al. 2026 (submitted Nature)", "arXiv:2603.07187"),
    ("3I/ATLAS",     "nucleus_size_m",          "≈1000",  "m",     "Forbes & Butler 2025",             "arXiv:2512.18341"),
    ("3I/ATLAS",     "metal_emission_at_distance", "yes (Ni I, Fe I persist to r_h~2 AU)", "—",
                                                                  "Hutsemékers et al. 2026 UVES+VLT",  "arXiv:2605.07652"),
    ("3I/ATLAS",     "cosmic_ray_surface_processing", "yes", "—", "Maggiolo et al. 2025",              "arXiv:2510.26308"),
]

comparison_long = pd.DataFrame(
    COMPARISON_ROWS,
    columns=["object", "metric", "value", "unit", "source_label", "source_arxiv_or_doi"],
)

# Pivot to a human-readable wide table (values only; provenance preserved in long form)
comparison_wide = comparison_long.pivot_table(
    index="metric",
    columns="object",
    values="value",
    aggfunc="first",
).reindex(columns=["1I/'Oumuamua", "2I/Borisov", "3I/ATLAS"])

# Persist both forms — long with full provenance, wide for portal display
LONG_CSV = os.path.join(OUT_DIR, "interstellar_comparison_long.csv")
WIDE_CSV = os.path.join(OUT_DIR, "interstellar_comparison.csv")
comparison_long.to_csv(LONG_CSV, index=False)
comparison_wide.to_csv(WIDE_CSV)

print(f"Long-format rows: {len(comparison_long)}  ->  {LONG_CSV}")
print(f"Wide-format pivoted to {comparison_wide.shape[0]} metrics × {comparison_wide.shape[1]} objects  ->  {WIDE_CSV}")
comparison_wide


### D/H ratio in cometary water — placing 3I/ATLAS in context

Deuterium-to-hydrogen ratios in cometary water are a fossil signature of the temperature at which the ice formed: colder environments preserve more deuterium-enriched H₂O because the gas-phase exchange that drives the ratio toward the protosolar value runs more slowly. Across decades of Solar-System cometary measurements (Lis et al. 2019; Altwegg et al. 2015; Hartogh et al. 2011), D/H spans roughly:

- **Protosolar nebula:** ~2 × 10⁻⁵ (Geiss & Gloeckler 1998)
- **Earth ocean (VSMOW):** 1.557 × 10⁻⁴
- **Jupiter-family comets:** typically 1.5 – 3 × 10⁻⁴
- **Oort-cloud comets:** typically 2 – 5 × 10⁻⁴
- **Cold prestellar / ISM water:** > 1 × 10⁻³

The published lower limit on 3I/ATLAS water D/H from ALMA (Salazar Manzano et al. 2026, *Nature Astronomy* accepted) is **> 6.6 × 10⁻³** — higher than typical ISM values and an order of magnitude above any Solar-System comet. This is the headline number: it says 3I/ATLAS preserves chemistry from an environment **colder** than even the coldest known formation regions in our own system.

The chart below places the three interstellar objects (where measured) on the log-D/H axis alongside these reference families. The computation is just the plot — every number is a published measurement, but the *placement of an interstellar comet against the local family-tree* is what we add here.


In [ ]:
# D/H comparison chart: 3I/ATLAS against Solar-System cometary families
# Every reference value below is a published measurement; this cell adds the
# synthesis (placing all values on one log axis) but does not invent numbers.

import os
import numpy as np
import matplotlib.pyplot as plt

OUT_DIR = "outputs_3I_ATLAS"
FIG_DH = os.path.join(OUT_DIR, "dh_ratio_comparison.png")

# Reference points: (label, value, kind) — kind controls plot style
# kind: 'point' for a single measurement, 'range' for a family span
DH_REFERENCES = [
    ("Protosolar nebula",         2.1e-5, 2.1e-5, "point", "Geiss & Gloeckler 1998"),
    ("Earth ocean (VSMOW)",       1.557e-4, 1.557e-4, "point", "IAEA reference"),
    ("Jupiter-family comets",     1.5e-4, 3.0e-4, "range", "Lis et al. 2019; Altwegg+ 2015"),
    ("Oort-cloud comets",         2.0e-4, 5.0e-4, "range", "Lis et al. 2019"),
    ("Cold prestellar / ISM H2O", 1.0e-3, 3.0e-3, "range", "Cleeves et al. 2014"),
]

# Interstellar object measurements
ISO_MEASUREMENTS = [
    # (label, value, lower_limit_flag, source)
    ("2I/Borisov (limit)",  3.0e-4, False, "Aravind et al. — best constraint"),
    ("3I/ATLAS (D/H > 6.6e-3)", 6.6e-3, True,  "Salazar Manzano et al. 2026 arXiv:2603.07026"),
]

fig, ax = plt.subplots(figsize=(11, 5.5))
ax.set_xscale("log")
ax.set_xlim(1e-5, 1e-2)

# Plot reference families as horizontal bars
y_pos = 0
y_labels = []
for label, lo, hi, kind, src in DH_REFERENCES:
    if kind == "point":
        ax.plot([lo], [y_pos], marker="o", markersize=9, color="#4a90d9", zorder=3)
    else:
        ax.plot([lo, hi], [y_pos, y_pos], linewidth=8, color="#4a90d9",
                solid_capstyle="round", alpha=0.85, zorder=3)
    y_labels.append(label)
    y_pos += 1

# Plot ISO measurements with distinct colour
for label, val, is_limit, src in ISO_MEASUREMENTS:
    color = "#e07a3c" if not is_limit else "#cf3a3a"
    marker = "o" if not is_limit else ">"   # right-arrow marker for a lower limit
    ax.plot([val], [y_pos], marker=marker, markersize=14, color=color,
            markeredgecolor="black", markeredgewidth=0.7, zorder=4)
    y_labels.append(label)
    y_pos += 1

ax.set_yticks(range(len(y_labels)))
ax.set_yticklabels(y_labels, fontsize=10)
ax.set_xlabel("D/H ratio in H₂O (log scale)", fontsize=11)
ax.set_title("3I/ATLAS water D/H placed against Solar-System cometary families and ISM water\n"
             "Lower limit from ALMA (Salazar Manzano et al. 2026) exceeds all known reference points",
             fontsize=11)
ax.grid(True, which="both", axis="x", alpha=0.3)
ax.invert_yaxis()  # families on top, ISOs on bottom for emphasis

# Annotate ISM band as the previous "coldest known" reference
ax.axvspan(1e-3, 3e-3, alpha=0.08, color="#4a90d9", zorder=1)
ax.text(1.7e-3, len(y_labels) - 0.5,
        "previous 'coldest' bracket\n(cold prestellar / ISM)",
        fontsize=8, color="#3a6fa6", ha="center", va="center")

plt.tight_layout()
plt.savefig(FIG_DH, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {FIG_DH}")


## Part C — What the popular press got wrong (and what it got right)

A widely-circulated article (MSN, May 2026) presented five claims about 3I/ATLAS as a single "baffling mystery" package. The five claims are not equivalent in scientific weight. The table below grades each against the primary literature.

The verdict pattern is deliberate: **`confirmed` / `partly confirmed` / `wrong` / `unsupported`**. We do not use softer language. If a claim is contradicted by three independent null searches, we say so plainly with the arXiv IDs of all three.


In [ ]:
# Sourced audit of five popular-press claims
# Each row: (claim, verdict, evidence, primary_source)
# Verdicts: confirmed | partly confirmed | wrong | unsupported

import os
import pandas as pd

OUT_DIR = "outputs_3I_ATLAS"
AUDIT_CSV = os.path.join(OUT_DIR, "popular_press_audit.csv")

AUDIT_ROWS = [
    (
        "CO2-dominated coma rather than H2O",
        "confirmed",
        "JWST NIRSpec and MIRI both show elevated CO2 relative to H2O at r_h ≈ 2.2–2.5 AU. Independently confirmed by Subaru [O I] forbidden-line ratio at r_h = 2.87 AU and by SPHEREx pre-perihelion spectrophotometric mapping.",
        "Roth et al. 2026 (arXiv:2603.20460); Belyakov et al. 2026 ApJL 1001 L11 (arXiv:2601.22034); Shinnaka et al. 2026 (arXiv:2603.25002); Lisse et al. 2026 (arXiv:2512.07318)",
    ),
    (
        "Nickel emission far from the Sun where metal should not vaporize",
        "confirmed (with context)",
        "Ni I and Fe I detected by UVES+VLT shortly after perihelion and persisting at r_h ≈ 2 AU. This extends the Solar-System trend (Manfroid et al. 2021 Nature) where neutral metal emission appears at heliocentric distances where simple sublimation models forbid it. The puzzle predates 3I/ATLAS; this object is the first interstellar example.",
        "Hutsemékers, Manfroid, Opitom et al. 2026 (arXiv:2605.07652); Zhao et al. 2026 (arXiv:2603.07718)",
    ),
    (
        "Glowing bright green despite lacking carbon-chain molecules that typically create that color",
        "unsupported",
        "C2 (the carbon chain typically responsible for green coma colour), C3, CN, and CH have all been detected and quantified in 3I/ATLAS by multiple groups. Opitom et al. measured 12C/13C and 14N/15N isotope ratios in CN — meaning CN is present at normal-comet levels, not depleted. The article appears to conflate 'unusual coma' with 'no carbon chains', which is not what any published spectrum shows.",
        "Zhao et al. 2026 (arXiv:2603.07718); Opitom et al. 2026 submitted Nature (arXiv:2603.07187)",
    ),
    (
        "Trajectory has remained almost perfectly steady with minimal non-gravitational acceleration",
        "wrong",
        "Non-gravitational acceleration has been detected, measured, and modelled by multiple independent groups. Spada, Królikowska & Dones present a detailed systematic + statistical uncertainty budget. Thoss, Loeb & Burkert and Forbes & Butler use the NGA to infer a ~1 km diameter nucleus consistent with ordinary outgassing physics. Scarmato & Loeb link the NGA components to observed jet position angles. The NGA is small relative to the strong outgassing only because the nucleus is dense and well-collimated, not because the trajectory is anomalously steady.",
        "Spada, Królikowska & Dones 2026 (arXiv:2603.00782); Thoss, Loeb & Burkert 2026 (arXiv:2603.15735); Forbes & Butler 2025 (arXiv:2512.18341); Scarmato & Loeb 2026 (arXiv:2604.18773); Ahuja & Ganesh 2026 RNAAS (arXiv:2601.17083)",
    ),
    (
        "Path traces back toward the same region of space as the 1977 WOW signal",
        "unsupported and actively contradicted",
        "No peer-reviewed publication supports a dynamical association. The WOW signal (1977) is a single radio burst with no measured distance, no parallax, and no proper motion — its sky direction alone cannot define a 3-D point of origin. Three independent technosignature searches dedicated to 3I/ATLAS returned NULL results: Breakthrough Listen at the Green Bank Telescope (1–12 GHz), the Allen Telescope Array, and FAST. The 'WOW signal connection' is a popular-press artefact, not a literature finding.",
        "Jacobson-Bell et al. 2025 GBT (arXiv:2512.19763); Sheikh et al. 2025 ATA (arXiv:2512.18142); Li, Tao & Zhang 2026 FAST (arXiv:2603.19023)",
    ),
]

audit_df = pd.DataFrame(AUDIT_ROWS, columns=["claim", "verdict", "evidence", "primary_source"])
audit_df.to_csv(AUDIT_CSV, index=False)

# Pretty-print verdict summary
print("Verdict tally:")
print(audit_df["verdict"].value_counts().to_string())
print(f"\nFull audit saved to: {AUDIT_CSV}")

# Display columns trimmed for in-notebook readability; full text persists in the CSV
display_df = audit_df[["claim", "verdict"]].copy()
display_df


## What the literature actually says — the bigger story

Across roughly 100 papers on 3I/ATLAS submitted between July 2025 and May 2026, a coherent picture has emerged that is more interesting than the "baffling mystery" framing. The convergent conclusion is **cold, distant, primitive origin** — 3I/ATLAS preserves chemistry from a region of its parent system colder than the coldest known formation environment in our own.

The key evidence, with primary sources:

- **CO₂-dominated coma** — JWST NIRSpec and MIRI, Subaru forbidden-line constraint, SPHEREx mapping (arXiv:2603.20460, 2601.22034, 2603.25002, 2512.07318).
- **Enriched methane D/H = (3.31 ± 0.34)%** — Roth, Cordiner et al. 2026 JWST (arXiv:2603.20445).
- **Enriched water D/H > 6.6 × 10⁻³** — Salazar Manzano et al. 2026, *Nature Astronomy* accepted (arXiv:2603.07026). See the chart in Part B.
- **Anomalous CN isotope ratios** (¹²C/¹³C ≈ 147) — Opitom et al. 2026, submitted *Nature* (arXiv:2603.07187).
- **Cold-and-distant-origin synthesis** — Cordiner et al. 2026, in review at *Nature* (arXiv:2603.06911).
- **Galactic cosmic-ray surface processing** — Maggiolo et al. 2025 (arXiv:2510.26308).
- **Detected and characterised non-gravitational acceleration** consistent with ordinary outgassing — Spada, Królikowska & Dones (arXiv:2603.00782); Thoss, Loeb & Burkert (arXiv:2603.15735); Scarmato & Loeb (arXiv:2604.18773).
- **First imaging from Mars orbit** — Ren et al. 2026, *ApJL* accepted, China's Tianwen-1 (arXiv:2603.10350).
- **Three independent technosignature null results** — Breakthrough Listen / GBT (arXiv:2512.19763), Allen Telescope Array (arXiv:2512.18142), FAST (arXiv:2603.19023).

3I/ATLAS is not "behaving in ways scientists cannot explain." It is **the first interstellar object characterised across the full chemical-diagnostic spectrum**, and the diagnostics it returns are coherent and physically meaningful. That is the headline. Anyone telling a different story has a sourcing problem.

### Reproducibility

Re-running the four code cells of this post-publication update from a fresh kernel produces:

- `outputs_3I_ATLAS/interstellar_comparison_long.csv` — long-format curated literature snapshot (one row per metric × object), with `source_label` and `source_arxiv_or_doi` on every row.
- `outputs_3I_ATLAS/interstellar_comparison.csv` — pivoted wide table for human reading and portal embedding.
- `outputs_3I_ATLAS/dh_ratio_comparison.png` — D/H synthesis chart placing 3I/ATLAS against Solar-System cometary families and ISM water.
- `outputs_3I_ATLAS/popular_press_audit.csv` — five claims × four columns (claim, verdict, evidence, primary_source).

No external network calls are made in these cells. Every value is a published measurement vendored inline; if a reference becomes outdated, the update lands in one place.
